# OpenMC → pyMPPost collision file

Reads the OpenMC `tracks.h5` file and writes an MCNPX-PoliMi style collision file for pyMPPost.

pyMPPost can be downloaded from: https://gitlab.eecs.umich.edu/umich-dnng/pymppost

Neutrons and photons are always analysed separately: set `PARTICLE` in the configuration cell.

In [1]:
import os
import sys
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.colors import LogNorm

In [2]:
# Make the src/ folder importable
sys.path.append(os.path.abspath("/Users/loren/WORKSPACE/PHOTON/verification-and-validation/3-coupling-OpenMC-MPPost/src"))

import openmc
from TrackAnalyzer import build_analyzer, describe_history, particle_name

In [3]:
# ================================ CONFIGURATION ================================
PARTICLE         = "neutron"          # "neutron" | "photon"  (never both)
DETECTOR         = "stilbene"         # key of DETECTOR_LIBRARY
DETECTOR_MAT_IDS = [99]               # OpenMC material ID(s) of the detector


BASE_DIR   = "/Users/loren/WORKSPACE/PHOTON/verification-and-validation/3-coupling-OpenMC-MPPost"
CASE = "trans-stilbene/DD"
TRACK_DIR  = "/Volumes/PortableSSD/DATA/2026/UM-OPENMC-PYMPPOST-METHODOLOGY/" + CASE
TRACK_FILE = "OpenMC_tracks_4.0E5.h5"

# Incident energy below which events are not written [eV].
# For photons keep it >= the OpenMC photon energy cutoff (default 1 keV).
E_THRESHOLD_EV = {"neutron": 1e1, "photon": 1e3}

# Same meaning as the cards of the MCNPX-PoliMi input of this case:
# RPOL : (1) neutron, (2) photon thresholds on the deposited energy [MeV];
#        collisions that deposit less are not written, 0 = write every energy release
# IPOL_CELLS : cells for which collisions are written, as IPOL(7)-(8...) but with the
#        OpenMC cell IDs (IPOL(7) = number of cells in the list); None = every cell
RPOL       = (0.01, 100)
IPOL_CELLS = [20]

MAX_RESIDUAL   = 0.05     # neutrons: tolerance on |predicted - measured| E'/E
WRITE_COHERENT = False    # photons: write coherent scattering events (Edep = 0)
N_INSPECT      = 10       # number of initial histories printed in verbose mode

# Detector descriptions
#   neutron_targets: nuclide -> atomic weight ratio (A) and ZAID written in the file
#   photon_zaid    : representative ZAID written for photon interactions
DETECTOR_LIBRARY = {
    "stilbene": {
        "neutron_targets": {"H-1":  {"A": 0.99917, "zaid": 1001},
                            "C-12": {"A": 11.8969, "zaid": 6000}},
        "photon_zaid": 6000,
    },

    "helium4": {
        "neutron_targets": {"He-4":  {"A":3.968, "zaid": 2004}},
        "photon_zaid": 6000,
    },    
}
# ===============================================================================

RESULTS_DIR = os.path.join(BASE_DIR, "results", CASE)
TRACK_PATH  = os.path.join(TRACK_DIR, TRACK_FILE)
DET         = DETECTOR_LIBRARY[DETECTOR]

# Count the histories without loading the track data
with h5py.File(TRACK_PATH, "r") as fh:
    SOURCE_PARTICLES = sum(1 for key in fh.keys() if key.startswith("track_"))
print(f"Total source particles: {SOURCE_PARTICLES / 1e5}E5")


Total source particles: 4.0E5


# Analysis

In [4]:
# Optional: list every particle of one history (type, birth energy and position).
# For photons, check that electrons/positrons appear: the photoelectric and
# pair-production reconstruction relies on them.
describe_history(TRACK_PATH, index=9)

History #9 (track_1_1_10): 1 particle tracks
  P #0   neutron   states=5    E0=2.4500e+06 eV  r0=(0.0000, 0.0000, -1.0000)  E_end=2.1804e+06 eV


In [5]:
output_path = os.path.join(
    RESULTS_DIR, f"OpenMC_collision_file_{PARTICLE}_{DETECTOR}_{SOURCE_PARTICLES / 1e5}E5.d")

# Particle-specific options
extra = ({"max_residual": MAX_RESIDUAL} if PARTICLE == "neutron"
         else {"write_coherent": WRITE_COHERENT})

analyzer = build_analyzer(PARTICLE, DET,
                          Ethreshold=E_THRESHOLD_EV[PARTICLE],
                          polimi_file_path=output_path,
                          **extra)

n_hist = analyzer.process_file(TRACK_PATH,
                               target_materials=DETECTOR_MAT_IDS,
                               n_inspect=N_INSPECT)

print(f"\nProcessed histories: {n_hist}")
print(f"Collision file     : {output_path}")
analyzer.summary()


========== HISTORY #0 ==========
  [H #0 | P #0] Event at state 2 (code -99): ✓ H-1
    Energy: 2.450e+06 eV → 6.839e+05 eV | Edep = 1.766e+06 eV
    cos θ = 0.529 | E'/E = 0.279
    H-1      residual 6.24e-05  ✅
  ------------------------------------------------
  [H #0 | P #0] Event at state 3 (code -99): ✓ H-1
    Energy: 6.839e+05 eV → 5.527e+05 eV | Edep = 1.313e+05 eV
    cos θ = 0.899 | E'/E = 0.808
    H-1      residual 1.84e-04  ✅
    C-12     residual 1.75e-01
  ------------------------------------------------
  [H #0 | P #0] Event at state 4 (code -99): ✓ C-12
    Energy: 5.527e+05 eV → 5.450e+05 eV | Edep = 7.627e+03 eV
    cos θ = 0.917 | E'/E = 0.986
    C-12     residual 1.27e-06  ✅
    H-1      residual 1.45e-01
  ------------------------------------------------
  → [H #0 | P #0] End of track: 7 states, final energy 5.45e+05 eV

========== HISTORY #1 ==========
  → [H #1 | P #0] No valid events in materials [99].
  → [H #1 | P #0] End of track: 4 states, final energy 2